<a href="https://colab.research.google.com/github/JoeVonDahab/pharmacology-graph/blob/main/pure_knowledge_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

Using device: cuda


## 1. Load Data

In [ ]:
# Load all data sources
drug_effects = pd.read_pickle('drug_clinical_effects.pkl')
drug_nodes = pd.read_pickle('approved_small_molecule_drugs.pkl')
drugs_interactions = pd.read_pickle('drug_protein_interactions.pkl')
protein_nodes = pd.read_pickle('protein_nodes_with_embeddings_extended.pkl')
drug_therapeutic_classes = pd.read_pickle('drug_therapeutic_classes_atc.pkl')
mechanism_of_action = pd.read_pickle('drug_mechanism_of_action.pkl')
drug_warnings = pd.read_pickle('drug_warnings_adverse_effects.pkl')

print("Data loaded:")
print(f"  Drugs: {len(drug_nodes)}")
print(f"  Proteins: {len(protein_nodes)}")
print(f"  Drug-Effects: {len(drug_effects)}")
print(f"  Drug-Protein Interactions: {len(drugs_interactions)}")
print(f"  Therapeutic Classes: {len(drug_therapeutic_classes)}")
print(f"  Mechanism of Action: {len(mechanism_of_action)}")
print(f"  Drug Warnings: {len(drug_warnings)}")

Data loaded:
  Drugs: 1915414
  Proteins: 4040
  Drug-Effects: 59954
  Drug-Protein Interactions: 1783205
  Therapeutic Classes: 4569
  Mechanism of Action: 7256
  Drug Warnings: 2302


## 2. Build Entity Indices (Embedding Lookup Tables)

In [ ]:
# Build entity-to-index mappings for each entity type

# DRUGS - use drug_internal_id
drug_to_idx = {int(row['drug_internal_id']): i for i, row in drug_nodes.iterrows()}
idx_to_drug = {i: int(row['drug_internal_id']) for i, row in drug_nodes.iterrows()}
num_drugs = len(drug_to_idx)

# PROTEINS - use protein_internal_id
protein_to_idx = {int(row['protein_internal_id']): i for i, row in protein_nodes.iterrows()}
idx_to_protein = {i: int(row['protein_internal_id']) for i, row in protein_nodes.iterrows()}
num_proteins = len(protein_to_idx)

# EFFECTS - use effect_id
unique_effects = drug_effects[['effect_id', 'effect_name']].drop_duplicates()
effect_to_idx = {str(row['effect_id']): i for i, (_, row) in enumerate(unique_effects.iterrows())}
idx_to_effect = {i: str(row['effect_id']) for i, (_, row) in enumerate(unique_effects.iterrows())}
num_effects = len(effect_to_idx)

# TARGETS - from mechanism_of_action (filter NaN)
target_data = mechanism_of_action[mechanism_of_action['target_internal_id'].notna()][['target_internal_id', 'target_name']].drop_duplicates()
target_to_idx = {int(row['target_internal_id']): i for i, (_, row) in enumerate(target_data.iterrows())}
idx_to_target = {i: int(row['target_internal_id']) for i, (_, row) in enumerate(target_data.iterrows())}
num_targets = len(target_to_idx)

# WARNINGS - use warning_description
valid_warnings = drug_warnings[drug_warnings['warning_description'].notna()]['warning_description'].unique()
warning_to_idx = {str(desc): i for i, desc in enumerate(valid_warnings)}
idx_to_warning = {i: str(desc) for i, desc in enumerate(valid_warnings)}
num_warnings = len(warning_to_idx)

# THERAPEUTIC CLASSES (Level 4 only for simplicity)
level4_descriptions = drug_therapeutic_classes['level4_description'].unique()
therapeutic_to_idx = {str(desc): i for i, desc in enumerate(level4_descriptions)}
idx_to_therapeutic = {i: str(desc) for i, desc in enumerate(level4_descriptions)}
num_therapeutics = len(therapeutic_to_idx)

print("\n" + "="*60)
print("ENTITY COUNTS (Embedding Lookup Tables)")
print("="*60)
print(f"  Drugs:       {num_drugs:,}")
print(f"  Proteins:    {num_proteins:,}")
print(f"  Effects:     {num_effects:,}")
print(f"  Targets:     {num_targets:,}")
print(f"  Warnings:    {num_warnings:,}")
print(f"  Therapeutics (L4): {num_therapeutics:,}")
print(f"  TOTAL:       {num_drugs + num_proteins + num_effects + num_targets + num_warnings + num_therapeutics:,}")


ENTITY COUNTS (Embedding Lookup Tables)
  Drugs:       1,915,414
  Proteins:    4,040
  Effects:     2,178
  Targets:     1,518
  Warnings:    462
  Therapeutics (L4): 666
  TOTAL:       1,924,278


## 3. Build Knowledge Graph Edges (Triplets)

In [ ]:
# Define relation types
RELATIONS = {
    'drug_binds_protein': 0,
    'drug_causes_effect': 1,
    'drug_acts_on_target': 2,
    'drug_has_warning': 3,
    'drug_has_therapeutic': 4,
}
num_relations = len(RELATIONS)

# Build triplets: (head_type, head_idx, relation, tail_type, tail_idx)
triplets = []

# 1. Drug-Protein interactions
print("Building Drug-Protein triplets...")
for _, row in tqdm(drugs_interactions.iterrows(), total=len(drugs_interactions)):
    drug_id = int(row['drug_internal_id'])
    protein_id = int(row['protein_internal_id'])

    if drug_id in drug_to_idx and protein_id in protein_to_idx:
        triplets.append(('drug', drug_to_idx[drug_id], RELATIONS['drug_binds_protein'], 'protein', protein_to_idx[protein_id]))

# 2. Drug-Effect
print("Building Drug-Effect triplets...")
for _, row in tqdm(drug_effects.iterrows(), total=len(drug_effects)):
    drug_id = int(row['drug_internal_id'])
    effect_id = str(row['effect_id'])

    if drug_id in drug_to_idx and effect_id in effect_to_idx:
        triplets.append(('drug', drug_to_idx[drug_id], RELATIONS['drug_causes_effect'], 'effect', effect_to_idx[effect_id]))

# 3. Drug-Target (Mechanism of Action)
print("Building Drug-Target triplets...")
valid_moa = mechanism_of_action[
    (mechanism_of_action['drug_internal_id'].notna()) &
    (mechanism_of_action['target_internal_id'].notna())
]
for _, row in tqdm(valid_moa.iterrows(), total=len(valid_moa)):
    drug_id = int(row['drug_internal_id'])
    target_id = int(row['target_internal_id'])

    if drug_id in drug_to_idx and target_id in target_to_idx:
        triplets.append(('drug', drug_to_idx[drug_id], RELATIONS['drug_acts_on_target'], 'target', target_to_idx[target_id]))

# 4. Drug-Warning
print("Building Drug-Warning triplets...")
valid_warn = drug_warnings[
    (drug_warnings['drug_internal_id'].notna()) &
    (drug_warnings['warning_description'].notna())
]
for _, row in tqdm(valid_warn.iterrows(), total=len(valid_warn)):
    drug_id = int(row['drug_internal_id'])
    warning_desc = str(row['warning_description'])

    if drug_id in drug_to_idx and warning_desc in warning_to_idx:
        triplets.append(('drug', drug_to_idx[drug_id], RELATIONS['drug_has_warning'], 'warning', warning_to_idx[warning_desc]))

# 5. Drug-Therapeutic Class
print("Building Drug-Therapeutic triplets...")
for _, row in tqdm(drug_therapeutic_classes.iterrows(), total=len(drug_therapeutic_classes)):
    drug_id = int(row['drug_internal_id'])
    therapeutic_desc = str(row['level4_description'])

    if drug_id in drug_to_idx and therapeutic_desc in therapeutic_to_idx:
        triplets.append(('drug', drug_to_idx[drug_id], RELATIONS['drug_has_therapeutic'], 'therapeutic', therapeutic_to_idx[therapeutic_desc]))

# Remove duplicates
triplets = list(set(triplets))

print(f"\n" + "="*60)
print("TRIPLET STATISTICS")
print("="*60)
relation_counts = {}
for t in triplets:
    rel = t[2]
    relation_counts[rel] = relation_counts.get(rel, 0) + 1

for rel_name, rel_idx in RELATIONS.items():
    count = relation_counts.get(rel_idx, 0)
    print(f"  {rel_name}: {count:,}")
print(f"  TOTAL: {len(triplets):,}")

Building Drug-Protein triplets...


100%|██████████| 1783205/1783205 [00:14<00:00, 124401.61it/s]


Building Drug-Effect triplets...


100%|██████████| 59954/59954 [00:00<00:00, 109930.86it/s]


Building Drug-Target triplets...


...

Building Drug-Warning triplets...


100%|██████████| 1012/1012 [00:00<00:00, 109431.67it/s]


Building Drug-Therapeutic triplets...


100%|██████████| 4569/4569 [00:00<00:00, 108767.05it/s]



TRIPLET STATISTICS
  drug_binds_protein: 961,097
  drug_causes_effect: 40,090
  drug_acts_on_target: 4,890
  drug_has_therapeutic: 3,404
  TOTAL: 1,009,995


## 4. TransR Model with Embedding Lookup

In [ ]:
class TransRKnowledgeGraph(nn.Module):
    """
    Pure TransR Knowledge Graph Embedding Model.

    Uses simple embedding lookup for all entities (no features).
    TransR projects entities to relation-specific spaces for scoring.

    Score(h, r, t) = -||M_r * h + r - M_r * t||_2
    """

    def __init__(self, entity_counts, num_relations, entity_dim=128, relation_dim=64, margin=1.0):
        super().__init__()

        self.entity_dim = entity_dim
        self.relation_dim = relation_dim
        self.margin = margin
        self.entity_counts = entity_counts

        # Entity embeddings (simple lookup) - one embedding table per entity type
        self.entity_embeddings = nn.ModuleDict({
            entity_type: nn.Embedding(count, entity_dim)
            for entity_type, count in entity_counts.items()
        })

        # Relation embeddings
        self.relation_embeddings = nn.Embedding(num_relations, relation_dim)

        # TransR projection matrices: project from entity_dim to relation_dim
        # M_r ∈ R^{relation_dim x entity_dim}
        self.projection_matrices = nn.Parameter(
            torch.zeros(num_relations, relation_dim, entity_dim)
        )

        # Initialize
        self._init_embeddings()

    def _init_embeddings(self):
        """Initialize embeddings with uniform distribution"""
        for emb in self.entity_embeddings.values():
            nn.init.uniform_(emb.weight, -6/np.sqrt(self.entity_dim), 6/np.sqrt(self.entity_dim))
            # Normalize
            emb.weight.data = F.normalize(emb.weight.data, p=2, dim=1)

        nn.init.uniform_(self.relation_embeddings.weight, -6/np.sqrt(self.relation_dim), 6/np.sqrt(self.relation_dim))
        # Normalize
        self.relation_embeddings.weight.data = F.normalize(self.relation_embeddings.weight.data, p=2, dim=1)

        # Initialize projection matrices as identity-like
        nn.init.eye_(self.projection_matrices.view(-1, self.relation_dim, self.entity_dim)[0])
        for i in range(1, self.projection_matrices.size(0)):
            nn.init.eye_(self.projection_matrices[i])

    def get_entity_embedding(self, entity_type, entity_idx):
        """Get embedding for entity by type and index"""
        return self.entity_embeddings[entity_type](entity_idx)

    def project_to_relation_space(self, entity_emb, relation_idx):
        """
        Project entity to relation-specific space.
        projected = M_r @ entity
        """
        # entity_emb: (batch, entity_dim)
        # M_r: (relation_dim, entity_dim)
        M_r = self.projection_matrices[relation_idx]  # (relation_dim, entity_dim)
        projected = torch.matmul(entity_emb, M_r.T)  # (batch, relation_dim)
        return F.normalize(projected, p=2, dim=-1)

    def score_triplet(self, head_type, head_idx, relation_idx, tail_type, tail_idx):
        """
        TransR scoring function.
        Score = -||M_r * h + r - M_r * t||_2
        """
        # Get entity embeddings
        h = self.get_entity_embedding(head_type, head_idx)  # (batch, entity_dim)
        t = self.get_entity_embedding(tail_type, tail_idx)  # (batch, entity_dim)

        # Project to relation space
        h_r = self.project_to_relation_space(h, relation_idx)  # (batch, relation_dim)
        t_r = self.project_to_relation_space(t, relation_idx)  # (batch, relation_dim)

        # Get relation vector
        r = self.relation_embeddings(torch.tensor([relation_idx], device=h.device))  # (1, relation_dim)

        # Compute distance: ||h_r + r - t_r||_2
        diff = h_r + r - t_r
        distance = torch.norm(diff, p=2, dim=-1)  # (batch,)

        return distance

    def forward(self, pos_triplets, neg_triplets):
        """
        Compute margin-based ranking loss.
        Loss = max(0, margin + pos_score - neg_score)

        pos_triplets: list of (head_type, head_idx, rel_idx, tail_type, tail_idx)
        neg_triplets: list of (head_type, head_idx, rel_idx, tail_type, tail_idx)
        """
        pos_scores = []
        neg_scores = []

        for pos, neg in zip(pos_triplets, neg_triplets):
            pos_score = self.score_triplet(
                pos[0], torch.tensor([pos[1]], device=next(self.parameters()).device),
                pos[2],
                pos[3], torch.tensor([pos[4]], device=next(self.parameters()).device)
            )
            neg_score = self.score_triplet(
                neg[0], torch.tensor([neg[1]], device=next(self.parameters()).device),
                neg[2],
                neg[3], torch.tensor([neg[4]], device=next(self.parameters()).device)
            )
            pos_scores.append(pos_score)
            neg_scores.append(neg_score)

        pos_scores = torch.cat(pos_scores)
        neg_scores = torch.cat(neg_scores)

        # Margin ranking loss (positive should have SMALLER distance)
        loss = F.relu(self.margin + pos_scores - neg_scores).mean()

        return loss

    def predict(self, head_type, head_idx, relation_idx, tail_type, tail_idx):
        """Predict probability of triplet being true (lower distance = higher prob)"""
        distance = self.score_triplet(head_type, head_idx, relation_idx, tail_type, tail_idx)
        # Convert distance to probability (sigmoid-based)
        prob = torch.sigmoid(self.margin - distance)
        return prob


# Initialize model
entity_counts = {
    'drug': num_drugs,
    'protein': num_proteins,
    'effect': num_effects,
    'target': num_targets,
    'warning': num_warnings,
    'therapeutic': num_therapeutics,
}

ENTITY_DIM = 128  # Embedding dimension for all entities
RELATION_DIM = 64  # Relation space dimension
MARGIN = 1.0

model = TransRKnowledgeGraph(
    entity_counts=entity_counts,
    num_relations=num_relations,
    entity_dim=ENTITY_DIM,
    relation_dim=RELATION_DIM,
    margin=MARGIN
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"\n" + "="*60)
print("MODEL SUMMARY")
print("="*60)
print(f"  Entity dimension: {ENTITY_DIM}")
print(f"  Relation dimension: {RELATION_DIM}")
print(f"  Margin: {MARGIN}")
print(f"  Total parameters: {total_params:,}")
print(f"  Memory: {total_params * 4 / 1e6:.2f} MB")


MODEL SUMMARY
  Entity dimension: 128
  Relation dimension: 64
  Margin: 1.0
  Total parameters: 246,348,864
  Memory: 985.40 MB


## 5. Dataset and DataLoader

In [ ]:
class KGTripletDataset(Dataset):
    """
    Dataset for Knowledge Graph triplets.
    Returns positive triplets and generates negative samples on-the-fly.
    """

    def __init__(self, triplets, entity_counts, neg_samples=1):
        self.triplets = triplets
        self.entity_counts = entity_counts
        self.neg_samples = neg_samples

        # Build set of positive triplets for filtering
        self.positive_set = set(triplets)

    def __len__(self):
        return len(self.triplets)

    def corrupt_triplet(self, triplet):
        """
        Generate negative sample by corrupting head or tail.
        """
        head_type, head_idx, rel, tail_type, tail_idx = triplet

        # Randomly corrupt head or tail
        if np.random.random() < 0.5:
            # Corrupt head
            new_head = np.random.randint(0, self.entity_counts[head_type])
            neg_triplet = (head_type, new_head, rel, tail_type, tail_idx)
        else:
            # Corrupt tail
            new_tail = np.random.randint(0, self.entity_counts[tail_type])
            neg_triplet = (head_type, head_idx, rel, tail_type, new_tail)

        # Make sure negative is not actually a positive
        max_tries = 10
        tries = 0
        while neg_triplet in self.positive_set and tries < max_tries:
            if np.random.random() < 0.5:
                new_head = np.random.randint(0, self.entity_counts[head_type])
                neg_triplet = (head_type, new_head, rel, tail_type, tail_idx)
            else:
                new_tail = np.random.randint(0, self.entity_counts[tail_type])
                neg_triplet = (head_type, head_idx, rel, tail_type, new_tail)
            tries += 1

        return neg_triplet

    def __getitem__(self, idx):
        pos_triplet = self.triplets[idx]
        neg_triplet = self.corrupt_triplet(pos_triplet)
        return pos_triplet, neg_triplet


def collate_triplets(batch):
    """Custom collate function for triplet batches"""
    pos_triplets = [item[0] for item in batch]
    neg_triplets = [item[1] for item in batch]
    return pos_triplets, neg_triplets


# Split data
np.random.seed(42)
indices = np.random.permutation(len(triplets))
train_size = int(0.8 * len(triplets))
val_size = int(0.1 * len(triplets))

train_triplets = [triplets[i] for i in indices[:train_size]]
val_triplets = [triplets[i] for i in indices[train_size:train_size+val_size]]
test_triplets = [triplets[i] for i in indices[train_size+val_size:]]

print(f"\nData split:")
print(f"  Train: {len(train_triplets):,}")
print(f"  Val:   {len(val_triplets):,}")
print(f"  Test:  {len(test_triplets):,}")

# Create datasets and dataloaders
train_dataset = KGTripletDataset(train_triplets, entity_counts)
val_dataset = KGTripletDataset(val_triplets, entity_counts)
test_dataset = KGTripletDataset(test_triplets, entity_counts)

BATCH_SIZE = 256

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_triplets, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_triplets, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_triplets, num_workers=0)


Data split:
  Train: 807,996
  Val:   100,999
  Test:  101,000


## 6. Training Loop

In [ ]:
def train_epoch(model, train_loader, optimizer, device):
    model.train()
    total_loss = 0

    for pos_batch, neg_batch in tqdm(train_loader, desc="Training"):
        optimizer.zero_grad()

        loss = model(pos_batch, neg_batch)
        loss.backward()

        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        # Normalize embeddings after update (important for TransR)
        with torch.no_grad():
            for emb in model.entity_embeddings.values():
                emb.weight.data = F.normalize(emb.weight.data, p=2, dim=1)
            model.relation_embeddings.weight.data = F.normalize(model.relation_embeddings.weight.data, p=2, dim=1)

        total_loss += loss.item()

    return total_loss / len(train_loader)


def evaluate(model, data_loader, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for pos_batch, neg_batch in data_loader:
            loss = model(pos_batch, neg_batch)
            total_loss += loss.item()

    return total_loss / len(data_loader)


# Training
EPOCHS = 50
LR = 0.001

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)

best_val_loss = float('inf')
history = {'train_loss': [], 'val_loss': []}

print("\n" + "="*60)
print("TRAINING")
print("="*60)

for epoch in range(EPOCHS):
    train_loss = train_epoch(model, train_loader, optimizer, DEVICE)
    val_loss = evaluate(model, val_loader, DEVICE)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)

    scheduler.step(val_loss)

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'transr_knowledge_embeddings_best.pt')

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")


TRAINING


Training: 100%|██████████| 3157/3157 [1:23:19<00:00,  1.58s/it]


Epoch 1/50 | Train Loss: 0.6213 | Val Loss: 0.5571 | LR: 0.001000


Training:  42%|████▏     | 1341/3157 [35:02<48:43,  1.61s/it] 

## 7. Evaluation Metrics

In [ ]:
import matplotlib.pyplot as plt

# Plot training history
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.plot(history['train_loss'], label='Train Loss')
ax.plot(history['val_loss'], label='Val Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('TransR Training Loss')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.savefig('transr_training_loss.png', dpi=150)
plt.show()

In [ ]:
def compute_metrics(model, test_triplets, entity_counts, device, sample_size=1000):
    """
    Compute link prediction metrics: Mean Rank, MRR, Hits@k
    """
    model.eval()

    # Sample for efficiency
    if len(test_triplets) > sample_size:
        sample_indices = np.random.choice(len(test_triplets), sample_size, replace=False)
        sampled_triplets = [test_triplets[i] for i in sample_indices]
    else:
        sampled_triplets = test_triplets

    ranks = []
    hits_at_1 = 0
    hits_at_3 = 0
    hits_at_10 = 0

    with torch.no_grad():
        for triplet in tqdm(sampled_triplets, desc="Computing metrics"):
            head_type, head_idx, rel, tail_type, tail_idx = triplet

            # Get true tail score
            true_score = model.score_triplet(
                head_type, torch.tensor([head_idx], device=device),
                rel,
                tail_type, torch.tensor([tail_idx], device=device)
            ).item()

            # Compare against all possible tails (sample for efficiency)
            num_candidates = min(100, entity_counts[tail_type])
            candidate_tails = np.random.choice(entity_counts[tail_type], num_candidates, replace=False)

            candidate_scores = []
            for cand_tail in candidate_tails:
                score = model.score_triplet(
                    head_type, torch.tensor([head_idx], device=device),
                    rel,
                    tail_type, torch.tensor([cand_tail], device=device)
                ).item()
                candidate_scores.append(score)

            # Rank (lower distance is better)
            rank = sum(1 for s in candidate_scores if s < true_score) + 1
            ranks.append(rank)

            if rank <= 1:
                hits_at_1 += 1
            if rank <= 3:
                hits_at_3 += 1
            if rank <= 10:
                hits_at_10 += 1

    n = len(sampled_triplets)
    mean_rank = np.mean(ranks)
    mrr = np.mean([1.0/r for r in ranks])

    print("\n" + "="*60)
    print("EVALUATION METRICS")
    print("="*60)
    print(f"  Mean Rank:  {mean_rank:.2f}")
    print(f"  MRR:        {mrr:.4f}")
    print(f"  Hits@1:     {hits_at_1/n*100:.2f}%")
    print(f"  Hits@3:     {hits_at_3/n*100:.2f}%")
    print(f"  Hits@10:    {hits_at_10/n*100:.2f}%")

    return {
        'mean_rank': mean_rank,
        'mrr': mrr,
        'hits_at_1': hits_at_1/n,
        'hits_at_3': hits_at_3/n,
        'hits_at_10': hits_at_10/n
    }

# Load best model and evaluate
model.load_state_dict(torch.load('transr_knowledge_embeddings_best.pt'))
metrics = compute_metrics(model, test_triplets, entity_counts, DEVICE)

## 8. Extract and Save Embeddings

In [ ]:
# Extract all embeddings
model.eval()

embeddings = {}
with torch.no_grad():
    for entity_type, emb_layer in model.entity_embeddings.items():
        embeddings[entity_type] = emb_layer.weight.cpu().numpy()
        print(f"  {entity_type}: {embeddings[entity_type].shape}")

# Save embeddings
np.savez('transr_entity_embeddings.npz', **embeddings)
print("\n✓ Saved entity embeddings to transr_entity_embeddings.npz")

# Save relation embeddings and projection matrices
relation_data = {
    'relation_embeddings': model.relation_embeddings.weight.detach().cpu().numpy(),
    'projection_matrices': model.projection_matrices.detach().cpu().numpy()
}
np.savez('transr_relation_data.npz', **relation_data)
print("✓ Saved relation data to transr_relation_data.npz")

## 9. Link Prediction Example

In [ ]:
def predict_tails(model, head_type, head_idx, relation_name, tail_type, entity_counts, top_k=10):
    """
    Predict most likely tail entities for a given head and relation.
    """
    model.eval()
    device = next(model.parameters()).device

    relation_idx = RELATIONS[relation_name]

    scores = []
    with torch.no_grad():
        for tail_idx in range(entity_counts[tail_type]):
            distance = model.score_triplet(
                head_type, torch.tensor([head_idx], device=device),
                relation_idx,
                tail_type, torch.tensor([tail_idx], device=device)
            ).item()
            scores.append((tail_idx, distance))

    # Sort by distance (lower is better)
    scores.sort(key=lambda x: x[1])

    return scores[:top_k]


# Example: Predict proteins that a drug might bind to
example_drug_idx = 0  # First drug
predictions = predict_tails(model, 'drug', example_drug_idx, 'drug_binds_protein', 'protein', entity_counts, top_k=10)

print("\n" + "="*60)
print(f"PREDICTION: Top 10 proteins for drug index {example_drug_idx}")
print("="*60)
for rank, (protein_idx, score) in enumerate(predictions, 1):
    print(f"  {rank}. Protein {protein_idx} (distance: {score:.4f})")

## 10. Save Full Model

In [ ]:
# Save full model state and config
save_dict = {
    'model_state_dict': model.state_dict(),
    'entity_counts': entity_counts,
    'num_relations': num_relations,
    'entity_dim': ENTITY_DIM,
    'relation_dim': RELATION_DIM,
    'margin': MARGIN,
    'relations': RELATIONS,
    'metrics': metrics,
    'history': history,
    # Mappings for inference
    'drug_to_idx': drug_to_idx,
    'protein_to_idx': protein_to_idx,
    'effect_to_idx': effect_to_idx,
    'target_to_idx': target_to_idx,
    'warning_to_idx': warning_to_idx,
    'therapeutic_to_idx': therapeutic_to_idx,
}

torch.save(save_dict, 'transr_full_model.pt')
print("\n✓ Saved full model to transr_full_model.pt")

print("\n" + "="*60)
print("TRAINING COMPLETE!")
print("="*60)
print(f"\nFiles saved:")
print(f"  • transr_full_model.pt (full model + config)")
print(f"  • transr_knowledge_embeddings_best.pt (best weights)")
print(f"  • transr_entity_embeddings.npz (numpy embeddings)")
print(f"  • transr_relation_data.npz (relation data)")
print(f"  • transr_training_loss.png (loss plot)")